# GPT-2 (124M) from Scratch in PyTorch
**12 layers | 12 heads | 768 dims | 1024 context | 50257 vocab**

Mixed precision, gradient accumulation, cosine LR, dual T4, HF Hub checkpointing.

In [ ]:
!pip install -q torch transformers datasets huggingface_hub

In [ ]:
import os, time, math, shutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import IterableDataset, DataLoader
from transformers import GPT2TokenizerFast
from datasets import load_dataset, interleave_datasets
from huggingface_hub import HfApi, hf_hub_download, login

CONFIG = dict(
    n_layer=12, n_head=12, n_embd=768,
    context_length=1024, vocab_size=50257,
    dropout=0.1, bias=True,
    batch_size=8, grad_accum=8,
    max_lr=6e-4, min_lr=6e-5,
    warmup_steps=2000, weight_decay=0.1,
    grad_clip=1.0, beta1=0.9, beta2=0.95,
    num_epochs=3, steps_per_epoch=25000,
    hf_repo_id='YOUR_USERNAME/gpt2-from-scratch',
    hf_token='YOUR_HF_TOKEN',
    log_interval=100,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device} | GPUs: {torch.cuda.device_count()}')

## Model Architecture

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg['n_embd'] % cfg['n_head'] == 0
        self.n_head = cfg['n_head']
        self.n_embd = cfg['n_embd']
        self.head_dim = cfg['n_embd'] // cfg['n_head']
        self.c_attn = nn.Linear(cfg['n_embd'], 3 * cfg['n_embd'], bias=cfg['bias'])
        self.c_proj = nn.Linear(cfg['n_embd'], cfg['n_embd'], bias=cfg['bias'])
        self.attn_drop = nn.Dropout(cfg['dropout'])
        self.resid_drop = nn.Dropout(cfg['dropout'])
        self.register_buffer('mask',
            torch.tril(torch.ones(cfg['context_length'], cfg['context_length']))
                 .view(1, 1, cfg['context_length'], cfg['context_length']))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))
        att = att.masked_fill(self.mask[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.c_proj(y))


class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.c_fc   = nn.Linear(cfg['n_embd'], 4 * cfg['n_embd'], bias=cfg['bias'])
        self.gelu   = nn.GELU()
        self.c_proj = nn.Linear(4 * cfg['n_embd'], cfg['n_embd'], bias=cfg['bias'])
        self.drop   = nn.Dropout(cfg['dropout'])

    def forward(self, x):
        return self.drop(self.c_proj(self.gelu(self.c_fc(x))))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1  = nn.LayerNorm(cfg['n_embd'])
        self.attn = CausalSelfAttention(cfg)
        self.ln2  = nn.LayerNorm(cfg['n_embd'])
        self.mlp  = MLP(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.transformer = nn.ModuleDict(dict(
            wte  = nn.Embedding(cfg['vocab_size'], cfg['n_embd']),
            wpe  = nn.Embedding(cfg['context_length'], cfg['n_embd']),
            drop = nn.Dropout(cfg['dropout']),
            h    = nn.ModuleList([Block(cfg) for _ in range(cfg['n_layer'])]),
            ln_f = nn.LayerNorm(cfg['n_embd']),
        ))
        self.lm_head = nn.Linear(cfg['n_embd'], cfg['vocab_size'], bias=False)
        self.transformer.wte.weight = self.lm_head.weight  # weight tying
        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg['n_layer']))
        n_params = sum(p.numel() for p in self.parameters())
        print(f'Model parameters: {n_params/1e6:.2f}M')

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, 0.0, 0.02)
            if module.bias is not None: nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, 0.0, 0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        else:
            logits = self.lm_head(x[:, [-1], :])
            loss = None
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=50):
        for _ in range(max_new_tokens):
            idx_crop = idx if idx.size(1) <= self.cfg['context_length'] else idx[:, -self.cfg['context_length']:]
            logits, _ = self(idx_crop)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

print('Model classes defined')

## Streaming Dataset Pipeline

In [ ]:
class StreamingGPTDataset(IterableDataset):
    """Streams text from HF datasets, tokenizes on-the-fly, yields (x, y) chunks."""

    def __init__(self, tokenizer, context_length):
        self.tokenizer = tokenizer
        self.ctx = context_length

    def _load_streams(self):
        streams = []
        for name, kw in [
            ('Skylion007/openwebtext', {}),
            ('wikipedia', {'name': '20220301.en'}),
            ('bookcorpus/bookcorpus', {}),
        ]:
            try:
                ds = load_dataset(name, streaming=True, split='train',
                                  trust_remote_code=True, **kw)
                streams.append(ds)
                print(f'  + {name} loaded')
            except Exception as e:
                print(f'  - {name} failed: {e}')
        if not streams:
            raise RuntimeError('No datasets loaded!')
        return interleave_datasets(streams) if len(streams) > 1 else streams[0]

    def __iter__(self):
        dataset = self._load_streams()
        buffer = []
        for example in dataset:
            text = example.get('text', '')
            if not text:
                continue
            tokens = self.tokenizer.encode(text, add_special_tokens=False)
            buffer.extend(tokens)
            while len(buffer) >= self.ctx + 1:
                chunk = buffer[:self.ctx + 1]
                buffer = buffer[self.ctx + 1:]
                x = torch.tensor(chunk[:-1], dtype=torch.long)
                y = torch.tensor(chunk[1:],  dtype=torch.long)
                yield x, y

tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
print(f'Tokenizer loaded | Vocab size: {tokenizer.vocab_size}')

## HuggingFace Hub Checkpointing

In [ ]:
def save_checkpoint(model, optimizer, scaler, epoch, step, cfg):
    """Save checkpoint to HuggingFace Hub, delete previous epoch."""
    api = HfApi(token=cfg['hf_token'])
    repo_id = cfg['hf_repo_id']
    try:
        api.create_repo(repo_id, exist_ok=True, private=True)
    except Exception:
        pass
    save_dir = f'./ckpt_epoch_{epoch}'
    os.makedirs(save_dir, exist_ok=True)
    path = os.path.join(save_dir, 'checkpoint.pt')
    state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
    torch.save({
        'model': state,
        'optimizer': optimizer.state_dict(),
        'scaler': scaler.state_dict(),
        'epoch': epoch,
        'global_step': step,
        'config': cfg,
    }, path)
    api.upload_file(path_or_fileobj=path,
                    path_in_repo=f'epoch_{epoch}/checkpoint.pt',
                    repo_id=repo_id)
    print(f'  Uploaded epoch {epoch} to {repo_id}')
    if epoch > 0:
        try:
            api.delete_folder(path_in_repo=f'epoch_{epoch - 1}', repo_id=repo_id)
            print(f'  Deleted epoch {epoch - 1} from Hub')
        except Exception as e:
            print(f'  Could not delete old checkpoint: {e}')
    shutil.rmtree(save_dir, ignore_errors=True)


def load_checkpoint(model, optimizer, scaler, cfg):
    """Resume from latest HuggingFace Hub checkpoint."""
    api = HfApi(token=cfg['hf_token'])
    repo_id = cfg['hf_repo_id']
    try:
        files = api.list_repo_files(repo_id)
    except Exception:
        print('No HF repo found. Starting fresh.')
        return 0, 0
    epochs = []
    for f in files:
        if f.startswith('epoch_') and f.endswith('/checkpoint.pt'):
            try:
                epochs.append(int(f.split('/')[0].split('_')[1]))
            except ValueError:
                pass
    if not epochs:
        print('No checkpoints on Hub. Starting fresh.')
        return 0, 0
    latest = max(epochs)
    print(f'Downloading checkpoint epoch {latest}...')
    ckpt_path = hf_hub_download(repo_id=repo_id,
                                 filename=f'epoch_{latest}/checkpoint.pt',
                                 token=cfg['hf_token'])
    ckpt = torch.load(ckpt_path, map_location='cpu')
    if hasattr(model, 'module'):
        model.module.load_state_dict(ckpt['model'])
    else:
        model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scaler.load_state_dict(ckpt['scaler'])
    start_epoch = ckpt['epoch'] + 1
    step = ckpt['global_step']
    print(f'Resumed from epoch {latest}, global step {step}')
    return start_epoch, step

print('Checkpoint utilities defined')

## Learning Rate Schedule + Training Loop

In [ ]:
def get_lr(step, cfg):
    """Cosine decay with linear warmup."""
    warmup = cfg['warmup_steps']
    total  = cfg['num_epochs'] * cfg['steps_per_epoch']
    if step < warmup:
        return cfg['max_lr'] * step / warmup
    if step >= total:
        return cfg['min_lr']
    ratio = (step - warmup) / (total - warmup)
    coeff = 0.5 * (1.0 + math.cos(math.pi * ratio))
    return cfg['min_lr'] + coeff * (cfg['max_lr'] - cfg['min_lr'])


def train(cfg):
    login(token=cfg['hf_token'])

    # Model
    model = GPT(cfg).to(device)
    if torch.cuda.device_count() > 1:
        print(f'Using DataParallel with {torch.cuda.device_count()} GPUs')
        model = nn.DataParallel(model)

    # Optimizer: weight decay only on >= 2D params
    decay, no_decay = [], []
    for pn, p in model.named_parameters():
        if not p.requires_grad: continue
        (decay if p.dim() >= 2 else no_decay).append(p)
    optimizer = torch.optim.AdamW([
        {'params': decay,    'weight_decay': cfg['weight_decay']},
        {'params': no_decay, 'weight_decay': 0.0},
    ], lr=cfg['max_lr'], betas=(cfg['beta1'], cfg['beta2']))

    scaler = GradScaler()

    # Resume from HF Hub
    start_epoch, global_step = load_checkpoint(model, optimizer, scaler, cfg)

    # Dataset
    dataset = StreamingGPTDataset(tokenizer, cfg['context_length'])
    loader  = DataLoader(dataset, batch_size=cfg['batch_size'],
                         num_workers=2, pin_memory=True)

    n_gpu = max(torch.cuda.device_count(), 1)
    eff_batch = cfg['batch_size'] * cfg['grad_accum'] * n_gpu
    total_steps = cfg['num_epochs'] * cfg['steps_per_epoch']
    print(f'\nEffective batch size: {eff_batch}')
    print(f'Total optimiser steps: {total_steps:,}')
    print(f'Tokens per step: {eff_batch * cfg["context_length"]:,}\n')

    for epoch in range(start_epoch, cfg['num_epochs']):
        t0 = time.time()
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0
        micro = 0
        epoch_steps = 0

        for x, y in loader:
            x, y = x.to(device), y.to(device)
            with torch.cuda.amp.autocast(dtype=torch.float16):
                _, loss = model(x, y)
                if loss.dim() > 0: loss = loss.mean()
                loss_scaled = loss / cfg['grad_accum']
            scaler.scale(loss_scaled).backward()
            running_loss += loss.item()
            micro += 1

            if micro % cfg['grad_accum'] == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['grad_clip'])
                lr = get_lr(global_step, cfg)
                for pg in optimizer.param_groups: pg['lr'] = lr
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1
                epoch_steps += 1

                if global_step % cfg['log_interval'] == 0:
                    avg = running_loss / cfg['grad_accum']
                    elapsed = time.time() - t0
                    tok_s = (epoch_steps * cfg['grad_accum'] * cfg['batch_size']
                             * n_gpu * cfg['context_length']) / elapsed
                    print(f'Epoch {epoch} | Step {global_step:>7,} | '
                          f'Loss {avg:.4f} | LR {lr:.2e} | {tok_s/1e3:.1f}k tok/s')
                running_loss = 0.0

                if epoch_steps >= cfg['steps_per_epoch']:
                    break

        elapsed = time.time() - t0
        print(f'\nEpoch {epoch} done in {elapsed/60:.1f} min '
              f'({epoch_steps:,} steps, global {global_step:,})')
        save_checkpoint(model, optimizer, scaler, epoch, global_step, cfg)
        print()

    print('Training complete!')
    return model

## Run Training

In [ ]:
model = train(CONFIG)

## Text Generation

In [ ]:
def generate_text(model, prompt, max_tokens=200, temperature=0.8, top_k=50):
    raw = model.module if hasattr(model, 'module') else model
    raw.eval()
    tokens = tokenizer.encode(prompt)
    idx = torch.tensor([tokens], dtype=torch.long, device=device)
    with torch.no_grad():
        out = raw.generate(idx, max_tokens, temperature=temperature, top_k=top_k)
    return tokenizer.decode(out[0].tolist())

prompts = [
    'The meaning of life is',
    'In a distant galaxy, the last star',
    'Artificial intelligence will',
]
for p in prompts:
    print(f'\n{"="*60}')
    print(f'PROMPT: {p}')
    print(f'{"="*60}')
    print(generate_text(model, p))